In [2]:
import joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

TRAIN_CSV   = "/content/tickets_sintetico_6000.csv"
MODEL_DIR   = Path("model")
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

CHANNEL_W = {"Telefone": 3, "Chat": 2, "Email": 1}
TICKET_W  = {"Problema Técnico": 4, "Reclamação": 3, "Solicitação de reembolso": 2,
             "Cancelamento": 1, "Dúvida": 0}

_CRIT_ANCHORS = [
    "entretenimento, lazer, hobby, item recreativo não essencial para a rotina",
    "eletrodoméstico auxiliar, conforto pessoal, dispositivo secundário, não indispensável",
    "ferramenta de trabalho e produtividade, comunicação diária indispensável",
    "equipamento vital para alimentação, segurança ou saúde, risco de incêndio ou acidente",
]
_anchor_embs: np.ndarray | None = None


def _anchor_matrix(encoder: SentenceTransformer) -> np.ndarray:
    global _anchor_embs
    if _anchor_embs is None:
        _anchor_embs = encoder.encode(_CRIT_ANCHORS, normalize_embeddings=True)
    return _anchor_embs


def _product_risk(names: pd.Series, encoder: SentenceTransformer) -> np.ndarray:
    embs = encoder.encode(names.fillna("produto desconhecido").tolist(), normalize_embeddings=True)
    return (np.argmax(embs @ _anchor_matrix(encoder).T, axis=1) + 1).astype(np.float32)


def _struct(df: pd.DataFrame, encoder: SentenceTransformer) -> np.ndarray:
    return np.column_stack([
        _product_risk(df["Product_Purchased"], encoder),
        df["Ticket_Channel"].map(CHANNEL_W).fillna(1).astype(int),
        (df["Customer_Age"] >= 60).astype(int),
        df["Ticket_Description"].fillna("").apply(lambda x: len(x.split())),
        df["Ticket_Type"].map(TICKET_W).fillna(2).astype(int),
        df["Customer_Age"].fillna(30),
    ]).astype(np.float32)


def _build_X(df: pd.DataFrame, encoder: SentenceTransformer) -> np.ndarray:
    texts = (df["Ticket_Subject"].fillna("") + ". " + df["Ticket_Description"].fillna("")).tolist()
    emb   = encoder.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    return np.hstack([_struct(df, encoder), emb.astype(np.float32)])


def train() -> tuple:
    MODEL_DIR.mkdir(exist_ok=True)

    df = pd.read_csv(TRAIN_CSV).dropna(subset=["Ticket_Description", "Ticket_Priority"])
    le = LabelEncoder().fit(df["Ticket_Priority"])
    y  = le.transform(df["Ticket_Priority"])

    encoder = SentenceTransformer(EMBED_MODEL)
    X = _build_X(df, encoder)

    # 60 / 20 / 20 stratified split — single source, guarantees same distribution
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.40, stratify=y, random_state=42)
    X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

    # SMOTE strictly on the training partition — val and test see only real samples
    X_tr_bal, y_tr_bal = SMOTE(random_state=42, k_neighbors=3).fit_resample(X_tr, y_tr)

    model = XGBClassifier(
        n_estimators=600,
        max_depth=4,           # shallower trees — less memorization
        learning_rate=0.05,
        subsample=0.75,
        colsample_bytree=0.65, # sample 65% of 390 features per tree — key for high-dim data
        min_child_weight=8,    # leaf must cover ≥8 weighted samples before splitting
        reg_lambda=2.0,        # L2 weight regularization
        reg_alpha=0.2,         # L1 sparsity regularization
        eval_metric="mlogloss",
        early_stopping_rounds=40,
        n_jobs=-1,
        verbosity=0,
        random_state=42,
    )
    model.fit(X_tr_bal, y_tr_bal, eval_set=[(X_val, y_val)], verbose=False)

    val_f1 = f1_score(y_val, model.predict(X_val), average="macro")
    te_f1  = f1_score(y_te,  model.predict(X_te),  average="macro")
    print(f"Val  F1-Macro : {val_f1:.4f}")
    print(f"Test F1-Macro : {te_f1:.4f}  (gap: {abs(val_f1 - te_f1):.4f})")
    print(classification_report(y_te, model.predict(X_te), target_names=le.classes_))

    joblib.dump(model, MODEL_DIR / "model.pkl")
    joblib.dump(le,    MODEL_DIR / "label_encoder.pkl")
    return model, encoder, le


def load_artifacts() -> tuple:
    return (
        joblib.load(MODEL_DIR / "model.pkl"),
        SentenceTransformer(EMBED_MODEL),
        joblib.load(MODEL_DIR / "label_encoder.pkl"),
    )


def predict_new_ticket(ticket_data: dict, model, encoder: SentenceTransformer, le: LabelEncoder) -> dict:
    df    = pd.DataFrame([ticket_data])
    proba = model.predict_proba(_build_X(df, encoder))[0]
    idx   = int(np.argmax(proba))
    return {
        "priority":     le.classes_[idx],
        "confidence":   round(float(proba[idx]), 4),
        "distribution": {cls: round(float(p), 4) for cls, p in zip(le.classes_, proba)},
    }


if __name__ == "__main__":
    model, encoder, le = train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Val  F1-Macro : 0.8018
Test F1-Macro : 0.8164  (gap: 0.0145)
              precision    recall  f1-score   support

        Alta       0.83      0.92      0.87       433
       Baixa       0.82      0.84      0.83       305
       Média       0.79      0.69      0.74       334
     Urgente       0.88      0.77      0.82       128

    accuracy                           0.82      1200
   macro avg       0.83      0.81      0.82      1200
weighted avg       0.82      0.82      0.82      1200

